# Machine Learning Course  
## Lesson 04 — Underfitting and Overfitting

**Previous lesson:** Polynomial Regression  
**Next lesson:** L1 and L2 Regularization

### Lesson Focus

Model complexity, generalization, and the bias–variance trade-off.

### Learning Objectives

By the end of this notebook, learners should be able to:

- explain the central concepts presented in this lesson;
- implement the relevant method using Python;
- evaluate the resulting model or computation appropriately;
- interpret outputs rather than reporting values without context;
- identify common implementation and modeling mistakes.

### Prerequisites

Familiarity with Python, NumPy, pandas, Matplotlib, and the concepts introduced in the preceding lesson.

### Reproducibility Standard

All data splits and stochastic estimators should use a fixed random seed. Preprocessing must be learned from training data only, preferably through a pipeline.

## 1. Conceptual Foundation

A model **underfits** when it is too simple to capture the structure of the data. It performs poorly on both training and validation data.

A model **overfits** when it captures training-specific noise. It performs very well on training data but generalizes poorly.

The goal is not to minimize training error alone. The goal is to select a model complexity that minimizes expected error on unseen data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

## 2. Generate a Nonlinear Dataset

The true relationship is nonlinear, and random noise is added to represent measurement error and unobserved influences.

In [ ]:
X = np.sort(rng.uniform(-3, 3, size=120)).reshape(-1, 1)
y_true = 0.5 * X[:, 0] ** 3 - 2 * X[:, 0] + 1
y = y_true + rng.normal(0, 2.5, size=len(X))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE
)

plt.figure(figsize=(8, 5))
plt.scatter(X_train[:, 0], y_train, label="Training data")
plt.scatter(X_test[:, 0], y_test, label="Test data")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Synthetic Nonlinear Regression Data")
plt.legend()
plt.show()

## 3. Compare Model Complexities

Polynomial degree controls model flexibility. Degree 1 is linear; higher degrees create progressively more flexible functions.

In [ ]:
degrees = [1, 2, 3, 5, 10, 15]
records = []
fitted_models = {}

for degree in degrees:
    model = Pipeline(
        steps=[
            ("polynomial", PolynomialFeatures(degree=degree, include_bias=False)),
            ("scaler", StandardScaler()),
            ("regressor", LinearRegression()),
        ]
    )
    model.fit(X_train, y_train)
    fitted_models[degree] = model

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    records.append(
        {
            "degree": degree,
            "train_rmse": np.sqrt(mean_squared_error(y_train, train_pred)),
            "test_rmse": np.sqrt(mean_squared_error(y_test, test_pred)),
            "train_r2": r2_score(y_train, train_pred),
            "test_r2": r2_score(y_test, test_pred),
        }
    )

results = pd.DataFrame(records)
results

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(results["degree"], results["train_rmse"], marker="o", label="Training RMSE")
plt.plot(results["degree"], results["test_rmse"], marker="o", label="Test RMSE")
plt.xlabel("Polynomial Degree")
plt.ylabel("RMSE")
plt.title("Training and Test Error versus Model Complexity")
plt.xticks(degrees)
plt.legend()
plt.show()

### Interpretation

- High training and test error indicate underfitting.
- Low training error combined with much higher test error indicates overfitting.
- The preferred degree is selected using validation or cross-validation performance, not the test set in a production workflow.

In [ ]:
x_grid = np.linspace(X.min(), X.max(), 500).reshape(-1, 1)

plt.figure(figsize=(9, 6))
plt.scatter(X_train[:, 0], y_train, label="Training data")

for degree in [1, 3, 15]:
    plt.plot(
        x_grid[:, 0],
        fitted_models[degree].predict(x_grid),
        label=f"Degree {degree}",
    )

plt.xlabel("x")
plt.ylabel("Predicted y")
plt.title("Underfitting, Appropriate Complexity, and Overfitting")
plt.legend()
plt.show()

## 4. Bias–Variance Trade-off

Expected prediction error can be described conceptually as the combination of:

- **Bias:** systematic error caused by restrictive assumptions;
- **Variance:** sensitivity to changes in the training sample;
- **Irreducible noise:** variation that the available features cannot explain.

Low-complexity models often have high bias and low variance. Excessively complex models often have low bias and high variance.

## 5. Practical Controls

Common ways to improve generalization include:

- collecting more representative data;
- reducing unnecessary features or polynomial degree;
- using cross-validation;
- applying L1 or L2 regularization;
- early stopping for iterative models;
- pruning trees or limiting their depth;
- simplifying neural-network architectures;
- preventing data leakage through pipelines.

## Lesson Review

### Key Takeaways

- The method should be understood through both its mathematical objective and its implementation.
- Evaluation must use data that were not used to fit the model.
- Preprocessing, model selection, and interpretation are part of the modeling workflow.
- Strong performance metrics do not replace error analysis or domain-aware interpretation.

### Exercises

1. Re-run the main experiment with a different random seed and compare the results.
2. Identify the most important modeling assumption in this lesson.
3. Add at least one additional evaluation metric or diagnostic visualization.
4. Explain one circumstance in which the demonstrated method would be unsuitable.
5. Convert the main workflow into a reusable scikit-learn pipeline where applicable.

### References

- scikit-learn user guide and API documentation
- NumPy and pandas documentation
- James, Witten, Hastie, and Tibshirani, *An Introduction to Statistical Learning*
- Géron, *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*

---

**Next lesson:** L1 and L2 Regularization